In [7]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1', engine='python', quotechar='"', on_bad_lines='skip')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
negative    2783
positive    2721
Name: count, dtype: int64


In [9]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})


In [30]:
negation_words = ['no', 'not', 'never', 'none', 'nobody', 'nowhere', 'nothing', 'neither', 'nor', 'hardly', 'scarcely', 'barely', 'don\'t', 'isn\'t', 'wasn\'t', 'shouldn\'t', 'wouldn\'t', 'couldn\'t', 'won\'t', 'can\'t', 'didn\'t']

def clean_text(text):
  text=text.lower()
  text= re.sub(r"[a-zA\s']"," ",text)
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" "," "))
  return text

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding='post'
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding='post'
)


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout = 0.3,recurrent_dropout = 0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 48s 780ms/step - accuracy: 0.5094 - loss: 0.6937 - val_accuracy: 0.4983 - val_loss: 0.6961
Epoch 2/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 43s 773ms/step - accuracy: 0.5622 - loss: 0.6783 - val_accuracy: 0.5778 - val_loss: 0.6698
Epoch 3/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 42s 749ms/step - accuracy: 0.6167 - loss: 0.5999 - val_accuracy: 0.5846 - val_loss: 0.6585
Epoch 4/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 43s 765ms/step - accuracy: 0.6519 - loss: 0.5308 - val_accuracy: 0.5301 - val_loss: 0.7429
Epoch 5/5
56/56 ━━━━━━━━━━━━━━━━━━━━ 82s 768ms/step - accuracy: 0.6434 - loss: 0.4984 - val_accuracy: 0.5619 - val_loss: 0.7932


In [33]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")

Test Accuracy: 57.86%


In [31]:
import re

def predict_sentiment(review):
    review = clean_text(review)

    seq = tokenizer.texts_to_sequences([review])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    prediction = model.predict(padded, verbose=0)[0][0]

    print("\nReview:", review)
    print("Score:", prediction)

    if prediction >= 0.5:
        print("Sentiment: Positive")
    else:
        print("Sentiment: Negative")


# Test the function
predict_sentiment("this was amazing")


Review:                 
Score: 0.48764494
Sentiment: Negative
